# Step 2 — Test SQLCoder via Ollama

Make sure `ollama serve` is running before executing these cells.

In [ ]:
import requests

OLLAMA_URL = 'http://localhost:11434'
MODEL      = 'sqlcoder:7b'

# Check ollama is running
try:
    r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama running ✅')
    print(f'Available models: {models}')
    if MODEL not in models:
        print(f'⚠️  {MODEL} not found — run: ollama pull {MODEL}')
    else:
        print(f'✅ {MODEL} ready')
except Exception as e:
    print(f'❌ Ollama not reachable: {e}')
    print('Run: ollama serve')

In [ ]:
import json

def query_sqlcoder(structured_input: str, timeout: int = 60) -> str:
    """Send a prompt to SQLCoder via Ollama and return the SQL."""
    prompt = (
        '### Task\n'
        'Generate a SQL query to answer the following question.\n\n'
        '### Input\n'
        f'{structured_input}\n\n'
        '### Response\n'
    )
    payload = {
        'model': MODEL,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': 0,
            'num_predict': 200,
        }
    }
    r = requests.post(f'{OLLAMA_URL}/api/generate', json=payload, timeout=timeout)
    return r.json()['response'].strip()

print('✅ query_sqlcoder() ready')

In [ ]:
# Sanity check — 3 samples from your test set BEFORE fine-tuning
import json
from pathlib import Path

test_samples = []
with open('data/test_inference.jsonl', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        test_samples.append(json.loads(line))

with open('data/test.jsonl', encoding='utf-8') as f:
    gold_samples = [json.loads(f.readline()) for _ in range(3)]

print('BASE MODEL OUTPUT (before fine-tuning):')
print('='*60)
for i, (sample, gold) in enumerate(zip(test_samples, gold_samples)):
    pred = query_sqlcoder(sample['structured_input'])
    print(f'\n--- Sample {i+1} [{gold["sql_type"]}] ---')
    print(f'INPUT : {sample["structured_input"][:80]}...')
    print(f'GOLD  : {gold["sql"]}')
    print(f'PRED  : {pred}')
    match = pred.strip().lower().rstrip(';') == gold['sql'].strip().lower().rstrip(';')
    print(f'MATCH : {"✅" if match else "❌"}')
print()
print('✅ Baseline done — paste output in chat')

In [ ]:
# Score base model on 200 test samples (quick baseline, not full 10k)
import re
from tqdm.notebook import tqdm

def normalise(sql):
    return re.sub(r'\s+', ' ', sql.lower().strip().rstrip(';')).strip()

def exact_match(pred, gold):
    return int(normalise(pred) == normalise(gold))

def token_f1(pred, gold):
    p, g = set(normalise(pred).split()), set(normalise(gold).split())
    if not p or not g: return 0.0
    common = p & g
    if not common: return 0.0
    pr, rc = len(common)/len(p), len(common)/len(g)
    return 2*pr*rc/(pr+rc)

N = 200   # quick baseline — increase to 10000 for full eval (will take hours)
infer_samples, gold_samples = [], []

with open('data/test_inference.jsonl') as fi, open('data/test.jsonl') as fg:
    for i, (li, lg) in enumerate(zip(fi, fg)):
        if i >= N: break
        infer_samples.append(json.loads(li))
        gold_samples.append(json.loads(lg))

em_scores, f1_scores = [], []
from collections import defaultdict
type_em = defaultdict(list)

for sample, gold in tqdm(zip(infer_samples, gold_samples), total=N, desc='Baseline eval'):
    pred = query_sqlcoder(sample['structured_input'])
    em   = exact_match(pred, gold['sql'])
    f1   = token_f1(pred, gold['sql'])
    em_scores.append(em)
    f1_scores.append(f1)
    type_em[gold['sql_type']].append(em)

import numpy as np
print(f'\nBASELINE RESULTS (n={N})')
print('='*40)
print(f'  Exact Match : {np.mean(em_scores):.1%}')
print(f'  Token F1    : {np.mean(f1_scores):.3f}')
print()
print(f"  {'TYPE':<14} {'EM':>6}")
print('  ' + '-'*22)
for t in sorted(type_em):
    print(f"  {t:<14} {np.mean(type_em[t]):>6.1%}")

# Save baseline for comparison in report later
import json
Path('outputs').mkdir(exist_ok=True)
with open('outputs/baseline_results.json', 'w') as f:
    json.dump({
        'n': N,
        'exact_match': round(float(np.mean(em_scores)), 4),
        'token_f1':    round(float(np.mean(f1_scores)), 4),
        'by_type':     {t: round(float(np.mean(v)), 4) for t, v in type_em.items()}
    }, f, indent=2)

print()
print('✅ Baseline saved → outputs/baseline_results.json')
print('Paste output in chat — next step is fine-tuning')